## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## This Notebook:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

In [2]:
# Run this line if not already Installed:
#!pip install langchain-huggingface sentence-transformers

In [6]:
# Imports
import os
import glob
import tiktoken
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

## Connecting to OpenAI API and HuggingFace:

In [7]:
MODEL = 'gpt-4.1-nano'
db_name = 'vector_db'

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_TOKEN')

openai = OpenAI()
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Part A: Dividing Documents into Chunks

In [10]:
# Counting Characters in all Documents:

knowledge_base_path = 'knowledge-base/**/*.md'
files = glob.glob(knowledge_base_path, recursive= True)
print(f'Found {len(files)} files in Knowledge Base.')

entire_knowledge_base = ''

for file_path in files:
    with open(file_path, 'r', encoding= 'utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += '\n\n'

print(f'Total Characters in Knowledge Base: {len(entire_knowledge_base)}')

Found 76 files in Knowledge Base.
Total Characters in Knowledge Base: 304434


In [12]:
# Counting Number of Tokens in All Documents:

encoding = tiktoken.encoding_for_model(model_name= MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f'Total tokens for {MODEL}: {token_count}')

Total tokens for gpt-4.1-nano: 63555


### Loading in Everything in the Knowledge Base Using Langchain's Loaders:

In [20]:
folders = glob.glob('knowledge-base/*')
documents = []
# print(folders)

for folder in folders:
    # Setting up Document-Type as Folder Name:
    doc_type = os.path.basename(folder)

    # Directory Loader Instance:
    loader = DirectoryLoader(folder, glob= '**/*.md', loader_cls= TextLoader, loader_kwargs= {'encoding': 'utf-8'})

    # Loading Data:
    folder_docs = loader.load()

    # Adding Everything to One List:
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc)

print(f'Loaded {len(documents)} documents.')

Loaded 76 documents.


In [21]:
documents[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a